# 03 — Volatility Forecasting

This notebook compares simple and parametric volatility forecasts and produces the monthly volatility input used by the portfolio backtest.

## 0. Setup and helper functions


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model


In [ ]:
default_layout = dict(
    template="plotly_white",
    height=500,
    margin=dict(t=70, l=60, r=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

In [ ]:
returns = pd.read_csv("data/returns.csv", index_col=0, parse_dates=True)

IS_END = "2017-12-31"
OOS_START = "2018-01-01"
HORIZON = 21
TRADING_DAYS = 252
CASH_PROXY = "SHY"
risk_model_assets = [asset for asset in returns.columns if asset != CASH_PROXY]

OUTPUT_DIR = Path("results/volatility_forecast")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def evaluate_vol_forecast(vol_forecast, realized_vol, eps=1e-8):
    """Compare volatility forecasts with the forward realized-volatility target."""
    common_index = vol_forecast.index.intersection(realized_vol.index)
    common_cols = vol_forecast.columns.intersection(realized_vol.columns)

    forecast = vol_forecast.loc[common_index, common_cols]
    realized = realized_vol.loc[common_index, common_cols]

    error = forecast - realized

    forecast_var = forecast.pow(2).clip(lower=eps)
    realized_var = realized.pow(2).clip(lower=eps)

    ratio = realized_var / forecast_var

    qlike = (
        ratio
        - np.log(ratio)
        - 1
    ).mean()

    summary = pd.DataFrame({
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "Bias": error.mean(),
        "Correlation": forecast.corrwith(realized),
        "QLIKE": qlike
    })

    return summary.round(4)


In [ ]:
def vol_forecast_regression_hac(vol_forecast, realized_vol, maxlags=21):
    """
    Run asset-level calibration regressions with HAC standard errors.

        realized_vol_t = alpha + beta * forecast_vol_t + error_t

    HAC errors are used because the 21-day realized-volatility target
    overlaps across adjacent forecast dates.
    """

    common_index = vol_forecast.index.intersection(realized_vol.index)
    common_cols = vol_forecast.columns.intersection(realized_vol.columns)

    results = []

    for asset in common_cols:
        data = pd.DataFrame({
            "realized": realized_vol.loc[common_index, asset],
            "forecast": vol_forecast.loc[common_index, asset]
        }).dropna()

        y = data["realized"]
        X = sm.add_constant(data["forecast"])

        model = sm.OLS(y, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": maxlags}
        )

        # Joint calibration check: zero intercept and unit forecast loading.
        joint_test = model.f_test("const = 0, forecast = 1")

        results.append({
            "Asset": asset,
            "Alpha": model.params["const"],
            "Beta": model.params["forecast"],
            "Alpha p-value": model.pvalues["const"],
            "Beta p-value": model.pvalues["forecast"],
            "R2": model.rsquared,
            "Joint F-stat": float(joint_test.fvalue),
            "Joint p-value": float(joint_test.pvalue),
        })

    return pd.DataFrame(results).set_index("Asset").round(4)


In [ ]:
def run_arch_lm_tests(returns_data, lags=5):
    """Run asset-level ARCH-LM tests on demeaned returns."""

    results = []

    for asset in returns_data.columns:
        r = returns_data[asset].dropna()

        # Demean returns before testing for ARCH effects.
        residuals = r - r.mean()

        lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(
            residuals,
            nlags=lags
        )

        results.append({
            "Asset": asset,
            "ARCH-LM Stat": lm_stat,
            "ARCH-LM p-value": lm_pvalue,
            "F-stat": f_stat,
            "F-test p-value": f_pvalue,
            "Lags": lags,
            "N obs": len(r)
        })

    return pd.DataFrame(results).set_index("Asset").round(4)


In [ ]:
def fit_garch(returns_data, vol="GARCH", p=1, o=0, q=1, dist="normal"):
    """Fit a univariate GARCH-family model for each asset and collect diagnostics."""

    models = {}
    results = []

    for asset in returns_data.columns:
        r = returns_data[asset].dropna()

        model = arch_model(
            r,
            mean="Constant",
            vol=vol,
            p=p,
            o=o,
            q=q,
            dist=dist,
            rescale=False
        )

        fitted = model.fit(disp="off")
        models[asset] = fitted

        params = fitted.params
        alpha = params.get("alpha[1]", np.nan)
        gamma = params.get("gamma[1]", np.nan)
        beta = params.get("beta[1]", np.nan)
        convergence_flag = getattr(fitted, "convergence_flag", np.nan)
        converged = bool(convergence_flag == 0) if pd.notna(convergence_flag) else np.nan

        row = {
            "Asset": asset,
            "Omega": params.get("omega", np.nan),
            "Alpha": alpha,
        }

        if o == 1:
            row.update({
                "Gamma": gamma,
                "Beta": beta,
                "Persistence": alpha + 0.5 * gamma + beta,
                "Gamma p-value": fitted.pvalues.get("gamma[1]", np.nan),
            })
        else:
            row.update({
                "Beta": beta,
                "Alpha + Beta": alpha + beta,
            })

        row.update({
            "Nu": params.get("nu", np.nan),
            "Converged": converged,
            "LogLik": fitted.loglikelihood,
            "AIC": fitted.aic,
            "BIC": fitted.bic
        })

        results.append(row)

    results = pd.DataFrame(results).set_index("Asset").round(4)

    return models, results


In [ ]:
def forecast_sanity_table(vol_forecast):
    """Summarize the distribution of forecast volatilities by asset."""
    return pd.DataFrame({
        "Min": vol_forecast.min(),
        "Median": vol_forecast.median(),
        "Mean": vol_forecast.mean(),
        "Max": vol_forecast.max()
    }).round(4)


In [ ]:
def garch_oos_forecast(
    returns_data,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t",
    is_end="2017-12-31",
    oos_start="2018-01-01",
    horizon=21
):
    """Generate OOS annualized volatility forecasts from fixed GARCH parameters."""
    forecasts = {}
    fitted_models = {}

    for asset in returns_data.columns:
        r = returns_data[asset].dropna()

        model = arch_model(
            r,
            mean="Constant",
            vol=model_type,
            p=p,
            o=o,
            q=q,
            dist=dist,
            rescale=False
        )

        fitted = model.fit(last_obs=is_end, disp="off")
        fitted_models[asset] = fitted

        forecast = fitted.forecast(
            start=oos_start,
            horizon=horizon,
            reindex=True,
            align="origin"
        )

        # Average expected daily variance across the forecast horizon.
        avg_var_forecast = forecast.variance.mean(axis=1)

        # Convert percent daily variance to annualized decimal volatility.
        forecasts[asset] = np.sqrt(avg_var_forecast) / 100 * np.sqrt(TRADING_DAYS)

    forecasts = pd.DataFrame(forecasts).loc[oos_start:]

    return forecasts.dropna(how="all"), fitted_models


## 1. Realized Volatility Target and Rolling Baseline

Forecasts are evaluated against a forward 21-trading-day realized-volatility target. For asset $i$ and forecast date $t$, the target uses returns from $t+1$ to $t+21$:

$$
RV_{i,t:t+21} = \sum_{j=1}^{21} r_{i,t+j}^{2}.
$$

The realized variance is converted to annualized realized volatility as:

$$
\sigma^{\mathrm{realized}}_{i,t:t+21}
=
\sqrt{\frac{252}{21} RV_{i,t:t+21}}.
$$

This target is ex-post and is used only for evaluation. Forecast inputs remain dated at $t$ or earlier; the target measures what is subsequently realized. Keeping these two objects separate avoids look-ahead in model construction and makes the timing of the evaluation explicit.


In [ ]:
realized_var_21d = (
    returns.pow(2)
    .rolling(window=HORIZON)
    .sum()
    .shift(-HORIZON)
)

realized_vol_21d = np.sqrt(realized_var_21d * TRADING_DAYS / HORIZON)

is_index = returns.loc[:IS_END].index
IS_EVAL_END = is_index[-HORIZON - 1]

realized_vol_is = realized_vol_21d.loc[:IS_EVAL_END]
realized_vol_oos = realized_vol_21d.loc[OOS_START:].dropna(how="all")


### Rolling Historical Volatility Baseline

The first benchmark is a 63-day rolling standard deviation of daily returns, annualized with 252 trading days. It is included as a transparent baseline for the model-selection exercise.

The estimate is shifted by one day, so the forecast at date $t$ uses only returns observed up to $t-1$. The same daily annualized volatility estimate is evaluated against the forward 21-day realized target, which treats the current risk estimate as the expected average risk level over the next month.

This benchmark is simple and stable, but it has two predictable drawbacks: it weights every observation in the window equally, and it reacts only after large moves enter the rolling window.


In [ ]:
ROLLING_WINDOW = 63

In [ ]:
rolling_vol = returns.rolling(window=ROLLING_WINDOW).std() * np.sqrt(TRADING_DAYS)
rolling_vol_lagged = rolling_vol.shift(1)
rolling_vol_lagged.tail()

In [ ]:
asset = "SPY"

vol_compare = pd.DataFrame({
    "Rolling 63D Forecast": rolling_vol_lagged[asset].loc[:IS_EVAL_END],
    "Forward 21D Realized Vol": realized_vol_is[asset]
}).dropna()

In [ ]:
fig = go.Figure()

for col in vol_compare.columns:
    fig.add_trace(
        go.Scatter(
            x=vol_compare.index,
            y=vol_compare[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title=f"{asset}: Rolling Volatility Forecast vs Forward Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    **default_layout
)

fig.show()

The SPY chart is shown for the in-sample period to keep the visual check separate from OOS evaluation. The rolling forecast captures the main volatility regimes, including 2008-2009 and the sovereign-credit stress period, but it reacts with a delay and remains elevated after shocks fade. That lag is the main weakness of a fixed-window estimator.


### Error Metrics

Forecasts are evaluated with MAE, RMSE, bias, forecast-realized correlation, and QLIKE. MAE and RMSE summarize error in volatility units, bias shows average over- or under-forecasting, correlation measures timing, and QLIKE evaluates forecasts in variance space. Lower MAE, RMSE, absolute bias, and QLIKE are preferred; higher correlation is preferred.


In [ ]:
evaluate_vol_forecast(rolling_vol_lagged.loc[:IS_EVAL_END], realized_vol_is)

The rolling benchmark carries useful signal: correlations are positive for every asset. Errors are lowest for short-duration fixed income and highest for assets exposed to larger regime shifts, such as EEM, VNQ, and SPY.

The positive bias indicates mild over-forecasting on average. For allocation, that is less concerning than persistent under-forecasting, but the delayed response around stress periods leaves room for a more adaptive benchmark.


## 2. EWMA / RiskMetrics Volatility Forecast

EWMA is added as a second benchmark. It keeps the historical-volatility logic but gives more weight to recent squared returns, allowing forecasts to update faster after market moves.

I use $\lambda = 0.94$, the standard daily RiskMetrics setting. The forecast is shifted by one day, so the date-$t$ estimate uses information available before $t$.


In [ ]:
LAMBDA_EWMA = 0.94

ewma_var = returns.pow(2).ewm(alpha=1 - LAMBDA_EWMA, adjust=False).mean()
ewma_vol = np.sqrt(ewma_var) * np.sqrt(TRADING_DAYS)

ewma_vol_lagged = ewma_vol.shift(1)

In [ ]:
asset = "SPY"

vol_compare_spy = pd.DataFrame({
    "Rolling 63D Forecast": rolling_vol_lagged[asset].loc[:IS_EVAL_END],
    "Forward 21D Realized Vol": realized_vol_is[asset],
    "EWMA Forecast": ewma_vol_lagged[asset].loc[:IS_EVAL_END]
}).dropna()

fig = go.Figure()

for col in vol_compare_spy.columns:
    fig.add_trace(
        go.Scatter(
            x=vol_compare_spy.index,
            y=vol_compare_spy[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title=f"{asset}: Volatility Forecasts vs Forward Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized Volatility",
    **default_layout
)

fig.show()

In [ ]:
evaluate_vol_forecast(ewma_vol_lagged.loc[:IS_EVAL_END], realized_vol_is)

For SPY, EWMA responds faster than the 63-day rolling estimate around 2008-2009. It still does not fully match the timing or size of realized volatility spikes, but it reduces the fixed-window lag.

The cross-asset table shows the same trade-off. EWMA improves responsiveness while remaining a benchmark rather than a final model: errors are still material for higher-volatility assets, and calibration has to be checked directly.


### Forecast Validation Regression

I use asset-level calibration regressions to check whether the forecast is informative and whether its scale is aligned with realized volatility:

$$
\sigma^{\mathrm{realized}}_{i,t:t+21}
=
\alpha_i
+
\beta_i \hat{\sigma}_{i,t}
+u_{i,t}.
$$

A positive $\beta_i$ indicates timing information. The joint test $\alpha_i = 0$ and $\beta_i = 1$ is a stricter calibration check. HAC/Newey-West standard errors are used because the 21-day realized target overlaps across adjacent dates.


In [ ]:
ewma_regression = vol_forecast_regression_hac(
    vol_forecast=ewma_vol_lagged.loc[:IS_EVAL_END],
    realized_vol=realized_vol_is,
    maxlags=HORIZON
)

ewma_regression

EWMA is informative in-sample: beta is positive and statistically significant for every asset. Explanatory power is strongest for VNQ, SHY, DBC, EEM, and EFA, and weaker for LQD and TLT.

The joint calibration test is rejected for most assets, so EWMA should not be treated as unbiased in level. It is a useful adaptive benchmark, but the fixed decay parameter leaves no room for asset-specific shock sensitivity or persistence.


## 3. GARCH Model Selection

The next step is to estimate asset-level GARCH specifications. The goal is not to overfit each ETF, but to test whether a common, interpretable volatility model is justified by the in-sample return dynamics.

Before fitting GARCH, I check for ARCH effects in returns. This is a diagnostic step: if returns show volatility clustering, conditional-volatility models are more defensible than a fixed-window estimate alone.


### ARCH-LM Test

The ARCH-LM test is used as a pre-check for volatility clustering. For each asset, demeaned returns are used to test whether lagged squared residuals explain current squared residuals:

$$
\hat{\varepsilon}_{i,t}^{2}
=
\alpha_0
+
\sum_{k=1}^{q} \alpha_k \hat{\varepsilon}_{i,t-k}^{2}
+u_{i,t}.
$$

The null is no ARCH effects. Rejection supports conditional-volatility modelling for that asset.


In [ ]:
returns_pct = returns.mul(100).dropna(how="all")
returns_pct_is = returns_pct.loc[:IS_END]
returns_pct_oos = returns_pct.loc[OOS_START:]

In [ ]:
arch_lm_results_5 = run_arch_lm_tests(
    returns_data=returns_pct_is,
    lags=5
)

arch_lm_results_5

The ARCH-LM results reject the no-ARCH null across the universe. This supports including conditional-volatility models in the comparison rather than relying only on rolling historical volatility.


In [ ]:
arch_lm_results_10 = run_arch_lm_tests(
    returns_data=returns_pct_is,
    lags=10
)

arch_lm_results_10

### Baseline Gaussian GARCH(1,1)

The baseline specification is Gaussian GARCH(1,1). It provides the reference point for the Student-t and GJR extensions:

$$
r_{i,t} = \mu_i + \varepsilon_{i,t},
\qquad
\varepsilon_{i,t} = \sigma_{i,t} z_{i,t}
$$

$$
\sigma_{i,t}^{2}
=
\omega_i
+
\alpha_i \varepsilon_{i,t-1}^{2}
+
\beta_i \sigma_{i,t-1}^{2}.
$$

For model selection, the key diagnostics are shock sensitivity ($\alpha$), persistence ($\beta$), total persistence ($\alpha + \beta$), likelihood-based criteria, and convergence behavior.


In [ ]:
garch_norm_models, garch_norm_results = fit_garch(returns_pct_is)

garch_norm_results

The Gaussian GARCH estimates show high persistence across the universe, with $alpha + beta$ close to one for most assets. That confirms that volatility shocks decay slowly and that a conditional-volatility model is appropriate.

The baseline is useful, but not final. Several assets sit very close to the persistence boundary, and Gaussian innovations may understate tail risk. That motivates testing Student-t innovations next.


### Student-t GARCH(1,1)

Student-t innovations are tested to allow fatter residual tails while keeping the same GARCH(1,1) variance recursion:

$$
\varepsilon_{i,t} = \sigma_{i,t} z_{i,t},
\qquad
z_{i,t} \sim t_{\nu_i}.
$$

This is a specification check: if information criteria improve without creating unstable estimates, the distributional assumption is better aligned with the return data.


In [ ]:
garch_t_models, garch_t_results = fit_garch(returns_pct_is, vol="GARCH", p=1, o=0, q=1, dist="t")

garch_t_results

The Student-t model raises a convergence warning for SHY, and its persistence is slightly above one. The information-criterion values for SHY are therefore not interpretable.

This is not a portfolio-wide failure. SHY is a low-volatility cash proxy, and the additional tail parameter is poorly identified for this series. I keep the Student-t comparison for the risk assets and treat SHY separately in the final forecast construction.


In [ ]:
garch_dist_comparison = pd.DataFrame({
    "Gaussian AIC": garch_norm_results["AIC"],
    "Student-t AIC": garch_t_results["AIC"],
    "AIC Improvement": garch_norm_results["AIC"] - garch_t_results["AIC"],
    "Gaussian BIC": garch_norm_results["BIC"],
    "Student-t BIC": garch_t_results["BIC"],
    "BIC Improvement": garch_norm_results["BIC"] - garch_t_results["BIC"],
}).round(4)

garch_dist_comparison.loc[risk_model_assets]

Excluding SHY, the information-criterion comparison favors Student-t innovations for every asset. The improvement is strongest for HYG, GLD, LQD, SPY, and EFA, which supports using a fat-tailed innovation distribution for the risk assets.

The conclusion should be stated carefully: Student-t improves in-sample likelihood fit, but it still has to prove useful in the out-of-sample forecast comparison.


### Testing Asymmetric Volatility Responses

The final candidate is GJR-GARCH with Student-t innovations. It tests whether negative shocks have a different impact on next-period volatility than positive shocks of the same size, which is most relevant for equity and credit assets:

$$
\sigma_{i,t}^{2}
=
\omega_i
+
\alpha_i \varepsilon_{i,t-1}^{2}
+
\gamma_i \varepsilon_{i,t-1}^{2} I(\varepsilon_{i,t-1}<0)
+
\beta_i \sigma_{i,t-1}^{2}.
$$

A positive $\gamma_i$ means negative shocks raise conditional volatility more than positive shocks of the same magnitude. The question is whether that extra parameter improves the model enough to justify using it in portfolio inputs.


In [ ]:
gjr_t_models, gjr_t_results = fit_garch(returns_pct_is, vol="GARCH", p=1, o=1, q=1, dist="t")

gjr_t_results

SHY again produces unstable GARCH estimates. I exclude SHY from the economic interpretation of the GJR comparison and keep it assigned to EWMA in the final construction.


In [ ]:
asymmetry_comparison = pd.DataFrame({
    "GARCH-t AIC": garch_t_results["AIC"],
    "GJR-t AIC": gjr_t_results["AIC"],
    "AIC Improvement": garch_t_results["AIC"] - gjr_t_results["AIC"],
    "GARCH-t BIC": garch_t_results["BIC"],
    "GJR-t BIC": gjr_t_results["BIC"],
    "BIC Improvement": garch_t_results["BIC"] - gjr_t_results["BIC"],
    "Gamma": gjr_t_results["Gamma"],
    "Gamma p-value": gjr_t_results["Gamma p-value"]
}).round(4)

asymmetry_comparison.loc[risk_model_assets]

The GJR results support asymmetry for the equity-sensitive assets and HYG: SPY, EEM, EFA, VNQ, and HYG have positive significant gamma estimates and better information criteria than symmetric Student-t GARCH.

The evidence is weaker for defensive and diversifying assets. IEF does not benefit from the asymmetric term, and DBC/TLT show statistical significance with limited information-criterion improvement. GLD has a small negative gamma, so the standard leverage interpretation does not apply cleanly.

This argues against selecting GJR automatically for the whole universe. It may fit some assets better in sample, but the final decision should be based on OOS forecast quality and robustness.


### Out-of-Sample Evaluation

The candidate GARCH specifications are evaluated on post-2018 forecasts against the same 21-day forward realized-volatility target. Parameters are estimated through the in-sample cutoff. These tables are validation diagnostics; they should not be described as a second round of strategy tuning.


In [ ]:
garch_t_oos_vol, garch_t_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="t",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [ ]:
gjr_t_oos_vol, gjr_t_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=1,
    q=1,
    dist="t",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [ ]:
garch_norm_oos_vol, garch_norm_oos_models = garch_oos_forecast(
    returns_data=returns_pct,
    model_type="GARCH",
    p=1,
    o=0,
    q=1,
    dist="normal",
    is_end=IS_END,
    oos_start=OOS_START,
    horizon=HORIZON
)

In [ ]:
garch_norm_oos_eval = evaluate_vol_forecast(
    vol_forecast=garch_norm_oos_vol,
    realized_vol=realized_vol_oos
)

garch_t_oos_eval = evaluate_vol_forecast(
    vol_forecast=garch_t_oos_vol,
    realized_vol=realized_vol_oos
)

gjr_t_oos_eval = evaluate_vol_forecast(
    vol_forecast=gjr_t_oos_vol,
    realized_vol=realized_vol_oos
)

### Forecast Error Metrics


In [ ]:
garch_oos_comparison = pd.DataFrame({
    "GARCH-N MAE": garch_norm_oos_eval["MAE"],
    "GARCH-t MAE": garch_t_oos_eval["MAE"],
    "GJR-t MAE": gjr_t_oos_eval["MAE"],

    "GARCH-N RMSE": garch_norm_oos_eval["RMSE"],
    "GARCH-t RMSE": garch_t_oos_eval["RMSE"],
    "GJR-t RMSE": gjr_t_oos_eval["RMSE"],

    "GARCH-N Corr": garch_norm_oos_eval["Correlation"],
    "GARCH-t Corr": garch_t_oos_eval["Correlation"],
    "GJR-t Corr": gjr_t_oos_eval["Correlation"],

    "GARCH-N QLIKE": garch_norm_oos_eval["QLIKE"],
    "GARCH-t QLIKE": garch_t_oos_eval["QLIKE"],
    "GJR-t QLIKE": gjr_t_oos_eval["QLIKE"]
}).round(4)

garch_oos_comparison.loc[risk_model_assets]

The OOS results should be read asset by asset rather than as a single ranking. The preferred specification varies by metric, and the more flexible models do not dominate consistently across the universe.

This matters for portfolio construction: a model with slightly better in-sample fit is not automatically the best allocation input. The selection should prioritize stable forecasts, acceptable OOS errors, and interpretable behavior across asset classes.


### Out-of-Sample Calibration Regressions


In [ ]:
garch_norm_oos_regression = vol_forecast_regression_hac(
    vol_forecast=garch_norm_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

garch_t_oos_regression = vol_forecast_regression_hac(
    vol_forecast=garch_t_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

gjr_t_oos_regression = vol_forecast_regression_hac(
    vol_forecast=gjr_t_oos_vol,
    realized_vol=realized_vol_oos,
    maxlags=HORIZON
)

In [ ]:
garch_oos_regression_comparison = pd.DataFrame({
    "GARCH-N Beta": garch_norm_oos_regression["Beta"],
    "GARCH-t Beta": garch_t_oos_regression["Beta"],
    "GJR-t Beta": gjr_t_oos_regression["Beta"],

    "GARCH-N R2": garch_norm_oos_regression["R2"],
    "GARCH-t R2": garch_t_oos_regression["R2"],
    "GJR-t R2": gjr_t_oos_regression["R2"],

    "GARCH-N Joint p-value": garch_norm_oos_regression["Joint p-value"],
    "GARCH-t Joint p-value": garch_t_oos_regression["Joint p-value"],
    "GJR-t Joint p-value": gjr_t_oos_regression["Joint p-value"]
}).round(4)

garch_oos_regression_comparison.loc[risk_model_assets]

The OOS calibration regressions show whether forecast levels move with realized volatility after 2018. Positive beta estimates indicate that the forecasts retain timing information. Slopes below one imply compressed forecast variation relative to the realized target, so the models pick up regimes but understate the size of some volatility moves.

The joint calibration tests are stricter than the error metrics and should not be expected to pass uniformly. Rejections mean the forecasts are useful conditional-risk estimates, not statistically unbiased forecasts of the realized-volatility proxy.

Together with the error metrics, this supports a conservative selection: added flexibility is useful only where it improves OOS behavior without creating convergence or stability problems.


## 4. Final Volatility Model Selection

The final volatility input uses Student-t GARCH(1,1) for the risk assets and EWMA/RiskMetrics for SHY.

The rationale is deliberately practical. ARCH-LM tests support conditional-volatility modelling, Student-t innovations improve in-sample fit for the risk assets, and a common symmetric GARCH specification is easier to defend than selecting a different model for every ETF. GJR-GARCH shows asymmetry for some equity- and credit-sensitive assets, but the extra parameter is not used in the final portfolio input because the benefit is uneven and the specification is less stable.

SHY is handled separately because its GARCH estimates show convergence and boundary problems. Using EWMA for the cash proxy avoids forcing an unstable parametric model onto a very low-volatility series.


In [ ]:
final_vol_forecast = garch_t_oos_vol.copy().loc[OOS_START:].dropna(how="all")

if CASH_PROXY in final_vol_forecast.columns:
    common_idx = final_vol_forecast.index.intersection(ewma_vol_lagged.index)
    final_vol_forecast.loc[common_idx, CASH_PROXY] = ewma_vol_lagged.loc[
        common_idx,
        CASH_PROXY
    ]

final_vol_forecast = final_vol_forecast.loc[OOS_START:].dropna(how="all")
forecast_sanity_table(final_vol_forecast)


The final forecast distribution is economically plausible. Equities, emerging markets, real estate, commodities, and gold carry higher average volatility, while high-quality fixed income and SHY remain lower-risk. TLT sits above shorter-duration Treasury exposure, consistent with its duration sensitivity.

SHY is handled separately because the Student-t and GJR estimates generated convergence and boundary issues. Using EWMA for SHY avoids forcing a parametric model onto a cash proxy where the GARCH likelihood is unstable.

The final dataset is an ex-ante OOS volatility panel: Student-t GARCH(1,1) for risk assets and EWMA/RiskMetrics for SHY.


In [ ]:
final_vol_forecast = final_vol_forecast.loc[OOS_START:].dropna(how="all")
final_vol_forecast.tail()

In [ ]:
# Month-end signal for the portfolio-allocation notebook.
# Use the latest available daily volatility forecast in each month.

final_vol_forecast_monthly = final_vol_forecast.resample("ME").last()
final_vol_forecast_monthly = final_vol_forecast_monthly.loc[OOS_START:].dropna(how="all")
final_vol_forecast_monthly.head()

In [ ]:
final_vol_forecast.to_csv("results/volatility_forecast/final_selected_vol_forecast_daily_oos.csv")
final_vol_forecast_monthly.to_csv("results/volatility_forecast/final_selected_vol_forecast_monthly_oos.csv")
